# KAT-VAD v2.1 — **E4-0: A3-ign × 5 seeds** and the base rule

Pre-registration: `core/docs/v2.1/KAT_VAD_PROPOSAL_v2.1.md` **§11.4** (base rule, operational definitions),
**§11.1** (endpoints, guardrails), **§7** (A3-ign predictions). A3-ign = A3 trained with
`loss.dvs_anchor_mode=ignore` — **nothing else changes** (the notebook proves it before any GPU time).
The control is **E3's five A3 runs** (`span`), read from `Thesis-V2/outputs/v2_e3/`, never retrained.

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps; refuses an upload without §11.4 or `e4_0_readout` | — |
| 1 | preflight: E3's A3/A0 runs + reads on Drive; **config diff = exactly `loss.dvs_anchor_mode`** for every seed; stage | no |
| 2 | **train** A3-ign × 5 (E3's recipe), chunked + resumable, J10 | **yes** |
| 3 | **diagnostics**: A3 (E3 ckpts) **and** A3-ign, same code (v2.1 `t2_val_reads`), O1′ vs E3's A0 | yes |
| 4 | **protocol B** for A3-ign on `dota_cap_dev` (A3's is E3's) | yes |
| 5 | **`e4_0_readout`** — the base rule → stop | no |

Nothing reads T2-test, DoTA-eval or DoTA-CAP-eval. A crashed run is re-run **with the same seed** — re-run the
cell; steps 2–4 skip whatever is already verified on Drive, so re-running the notebook resumes.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP, VideoMAE), v1 outputs
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # write: kat-vad/ (code), data/, cache/, outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V2_E4_0_ARM'), 'this upload predates E4-0 -- re-upload branch v2'
assert (REPO / 'core/tools/e4_0_readout.py').is_file(), 'upload has no core/tools/e4_0_readout.py'
PROPOSAL = REPO / 'core/docs/v2.1/KAT_VAD_PROPOSAL_v2.1.md'
assert '### 11.4 E4-0' in PROPOSAL.read_text(encoding='utf-8'), 'upload has no proposal v2.1 §11.4'

DATASET    = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
KERNEL     = constants.DADA_ORIGIN_SCORE_HEAD_KERNEL     # 3
TOPK_PCT   = constants.DADA_ORIGIN_MIL_TOPK_PCT          # 5
NUM_EPOCHS, CHUNK_EPOCHS = 20, 5                         # E3's recipe
SEEDS      = list(constants.V2_E3_SEEDS)                 # 2024-2028, paired with E3's A3
CONTROL    = constants.V2_E4_0_CONTROL                   # 'A3' (span)
ARM        = constants.V2_E4_0_ARM                       # 'A3ign'
ANCHOR     = constants.V2_E4_0_ANCHOR_MODE               # 'ignore'
CRN        = 'R2'
ENC        = constants.VIDEOMAE_ENCODER_S
MOTION_DIM = constants.VIDEOMAE_ARCH[ENC][0]             # 384
CAP_SPLIT  = constants.V2_SPLIT_DOTA_CAP_DEV

DOTA      = constants.DATA_ROOT / 'DoTA'
LABELS    = DOTA / 'labels_s8'                            # defs.json (class names)
DOTA_S1   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
V2_DATA   = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'
V2_CACHE  = Path(DRIVE_V2) / 'cache'
KNN_V2    = V2_CACHE / 'knn' / f'{DATASET}_v2' / constants.KNN_CACHE_FILENAME
BAKED_T2  = V2_CACHE / 'v2' / f'A3_{CRN}_S' / DATASET               # A3's T2 input cache (shared)
CAP_BAKED = V2_CACHE / 'v2' / f'A3_{CRN}_S' / 'DoTA_CAP_s3_from_s1'  # A3's DoTA-CAP input cache
E3        = constants.OUTPUT_ROOT / 'v2_e3'               # control: A3/s<seed>, A0 diags, REPORTS
E40       = constants.OUTPUT_ROOT / 'v2_e4_0'             # this step: A3ign/, A3/ (re-diag), REPORTS/
LOCAL = Path('/content/e4_0')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('V2_DATA', 'KNN_V2', 'BAKED_T2', 'CAP_BAKED', 'E3', 'E40'):
    print(f'{name:9s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- training will be unusably slow'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2)."""
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """Copy a VM-local result to Drive and fsync every file (C10)."""
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = [p for p in dst.rglob('*') if p.is_file()]
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        files = [dst]
    for p in files:
        with open(p, 'rb+') as fh:
            os.fsync(fh.fileno())


def ckpt_step(path):
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 1. Preflight + stage — the config diff is proven **before** any GPU time

In [ ]:
import tempfile

from core.config import load_config
from core.data.v2_splits import load_split
from core.tools import e3_readout, e4_0_readout, position_prior, protocol_b_eval, v2_dataset, v2_diagnostics

cap_dev = load_split(CAP_SPLIT)
assert len(cap_dev) == 569, len(cap_dev)
for path in (V2_DATA / v2_dataset.MANIFEST_FILENAME, KNN_V2, BAKED_T2, CAP_BAKED,
             LABELS / constants.DEFS_FILENAME,
             e3_readout.pb_dir(E3, CONTROL, CAP_SPLIT) / protocol_b_eval.CLIP_SCORES_NPZ,
             E3 / e3_readout.REPORTS / e3_readout.PRIOR_DIR / CAP_SPLIT / position_prior.PRIOR_NPZ):
    assert path.exists(), f'missing {path} (E3 must be on Drive)'


def verified(d):
    last = metrics_last(d / 'metrics.jsonl')
    return bool(last and last['epoch'] + 1 == NUM_EPOCHS
                and ckpt_step(d / 'checkpoint_last.pt') == last['global_step'])


def config_diff(a, b):
    out = set()
    for section in set(a) | set(b):
        sa, sb = a.get(section) or {}, b.get(section) or {}
        out |= {f'{section}.{k}' for k in set(sa) | set(sb) if sa.get(k) != sb.get(k)}
    return out


SETS = ['train.stage=2', f'train.num_epochs={NUM_EPOCHS}', 'train.amp=true', f'data.dataset={DATASET}',
        f'model.score_head_kernel={KERNEL}', f'loss.mil_topk_pct={TOPK_PCT}', 'kip.enabled=false',
        f'v2.crn={CRN}', f'v2.motion={ENC}', f'model.motion_dim={MOTION_DIM}',
        f'loss.dvs_anchor_mode={ANCHOR}']

E3_CFG = {}
for seed in SEEDS:
    d = E3 / CONTROL / f's{seed}'
    assert verified(d), f'E3 {CONTROL} s{seed} is not a finished run on Drive (J10)'
    a0 = E3 / 'A0' / f's{seed}' / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON
    assert a0.exists(), f'missing E3 A0 diag {a0} (O1′ reference)'
    E3_CFG[seed] = yaml.safe_load((d / 'config.yaml').read_text())
    assert E3_CFG[seed]['loss']['dvs_anchor_mode'] == 'span', f'E3 A3 s{seed} was not span'
    with tempfile.TemporaryDirectory() as tmp:     # the config A3-ign WILL train with, built now
        cfg = load_config(None, [*SETS, f'train.seed={seed}'])
        cfg.save_yaml(Path(tmp) / 'config.yaml')
        planned = yaml.safe_load((Path(tmp) / 'config.yaml').read_text())
    diff = config_diff(E3_CFG[seed], planned)
    assert diff == {'loss.dvs_anchor_mode'}, f's{seed}: A3-ign would differ from E3 A3 in {diff}'
print('every seed: A3-ign = E3 A3 + loss.dvs_anchor_mode=ignore, nothing else')

baked = json.loads((BAKED_T2 / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
print('A3 T2 cache train split', baked['train_ids_sha1'][:12])
L_V2 = stage(V2_DATA, LOCAL / 'stage' / 'corpus_v2')
L_KNN = stage(KNN_V2, LOCAL / 'stage' / 'knn_v2.npz')
L_T2 = stage(BAKED_T2, LOCAL / 'stage' / 'A3_T2')
L_CAP = stage(CAP_BAKED, LOCAL / 'stage' / 'A3_CAP')
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'DoTA_s1_ncc_cap', cap_dev)
L_DOTA_LABELS = stage(LABELS / constants.DEFS_FILENAME,
                      LOCAL / 'stage' / 'dota_labels' / constants.DEFS_FILENAME).parent

## 2. Train A3-ign × 5 — chunked, resumable, verified

E3's recipe plus `loss.dvs_anchor_mode=ignore`. After every chunk the written `config.yaml` is diffed against E3's
A3 of the same seed (must be exactly `loss.dvs_anchor_mode`). Skip/resume reads the checkpoint's own `global_step`
(J10). **Re-run this cell after a disconnect.**

In [ ]:
def run_dir(seed):
    return E40 / ARM / f's{seed}'


def train_cmd(seed, out_dir):
    return [sys.executable, '-m', 'core.train', *[x for s in [*SETS, f'train.seed={seed}'] for x in ('--set', s)],
            '--data-dir', L_V2, '--clip-dir', L_T2, '--knn-cache', L_KNN, '--output-dir', out_dir]


def sync(local, drive_dir):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        if (local / name).exists():
            publish(local / name, drive_dir / name)


def check_diff(seed, cfg_path):
    diff = config_diff(E3_CFG[seed], yaml.safe_load(cfg_path.read_text()))
    assert diff == {'loss.dvs_anchor_mode'}, f'A3-ign s{seed} differs from E3 A3 in {diff} -- STOP'


durations = []
todo = [s for s in SEEDS if not verified(run_dir(s))]
print(f'{len(todo)} runs to train in this session')
for k, seed in enumerate(todo):
    drive_dir, local = run_dir(seed), LOCAL / 'runs' / ARM / f's{seed}'
    local.mkdir(parents=True, exist_ok=True)
    last = metrics_last(drive_dir / 'metrics.jsonl')
    if not (local / 'checkpoint_last.pt').exists() and last and \
            ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step']:
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            shutil.copy2(drive_dir / name, local / name)          # resume from a consistent Drive copy
    t0 = time.time()
    while True:
        done = (metrics_last(local / 'metrics.jsonl') or {'epoch': -1})['epoch'] + 1
        if done >= NUM_EPOCHS:
            break
        cmd = train_cmd(seed, local)
        if done:
            cmd += ['--resume', local / 'checkpoint_last.pt']
        cmd += ['--stop-after-epochs', min(CHUNK_EPOCHS, NUM_EPOCHS - done)]
        print(f'=== {ARM} s{seed}: epochs {done} -> {min(done + CHUNK_EPOCHS, NUM_EPOCHS)} of {NUM_EPOCHS}')
        run(cmd, log=LOCAL / f'{ARM}_s{seed}_train.log', quiet_steps=True)
        check_diff(seed, local / 'config.yaml')
        sync(local, drive_dir)
    sync(local, drive_dir)
    assert verified(drive_dir), f'{ARM} s{seed}: J10 FAILS on Drive'
    check_diff(seed, drive_dir / 'config.yaml')
    durations.append(time.time() - t0)
    eta = sum(durations) / len(durations) * (len(todo) - k - 1)
    print(f'{ARM} s{seed}: step {metrics_last(drive_dir / "metrics.jsonl")["global_step"]}, '
          f'{durations[-1] / 60:.1f} min; ETA this session {eta / 3600:.1f} h')
missing = [s for s in SEEDS if not verified(run_dir(s))]
print('all 5 A3-ign runs verified' if not missing else f'still to train: {missing}')

## 3. Diagnostics — A3 (E3's checkpoints) and A3-ign, same code

E3's A3 diags predate `t2_val_reads` (normal-window peak, T2-val span profile), so A3 is **re-diagnosed** here from
its E3 checkpoints into `v2_e4_0/A3/`. O1′ reads E3's A0 diag of the same seed. O5 on `dota_cap_dev` (unlabelled).

In [ ]:
CKPT = {CONTROL: lambda s: E3 / CONTROL / f's{s}' / 'checkpoint_last.pt',
        ARM: lambda s: run_dir(s) / 'checkpoint_last.pt'}


def diagnose(arm, seed):
    out = E40 / arm / f's{seed}' / e3_readout.DIAG_DIR
    if (out / v2_diagnostics.DIAG_JSON).exists():
        return
    a0 = E3 / 'A0' / f's{seed}' / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON
    run([sys.executable, '-m', 'core.tools.v2_diagnostics', '--run', f'{arm}_s{seed}', CKPT[arm](seed),
         '--data-dir', L_V2, '--t2-input-dir', L_T2, '--dota-input-dir', L_CAP,
         '--dota-s1-dir', L_DOTA_S1, '--dota-data-dir', L_DOTA_LABELS, '--dota-split', CAP_SPLIT,
         '--a0-diag', a0, '--out-dir', out, '--device', 'auto'],
        log=LOCAL / f'{arm}_s{seed}_diag.log')


assert not [s for s in SEEDS if not verified(run_dir(s))], 'train every A3-ign run first'
for seed in SEEDS:
    for arm in (CONTROL, ARM):
        diagnose(arm, seed)
print('diagnosed:', sum((E40 / a / f's{s}' / e3_readout.DIAG_DIR / v2_diagnostics.DIAG_JSON).exists()
                        for a in (CONTROL, ARM) for s in SEEDS), 'of 10')

## 4. Protocol B for A3-ign on `dota_cap_dev`

All 5 checkpoints in one call, seed-averaged per frame; `per_run_macro` feeds the paired interval. A3's protocol-B
read is E3's (same checkpoints, same input cache) and is not recomputed.

In [ ]:
out = e3_readout.pb_dir(E40, ARM, CAP_SPLIT)
if (out / protocol_b_eval.CLIP_SCORES_NPZ).exists():
    print('reusing', out)
else:
    runs = [x for s in SEEDS for x in ('--run', f's{s}', run_dir(s) / 'checkpoint_last.pt')]
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *runs,
         '--input-dir', L_CAP, '--s1-dir', L_DOTA_S1,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_DOTA_LABELS, '--split', CAP_SPLIT, '--out-dir', out, '--device', 'auto'],
        log=LOCAL / f'{ARM}_{CAP_SPLIT}_pb.log')
print((out / protocol_b_eval.READOUT_MD).read_text())

## 5. Read-out and the base rule (§11.4) — no GPU

In [ ]:
run([sys.executable, '-m', 'core.tools.e4_0_readout', '--e3-dir', E3, '--e4-dir', E40])
print((E40 / e3_readout.REPORTS / e4_0_readout.READOUT_MD).read_text())

## 6. Record — then **stop**

Paste `v2_e4_0/REPORTS/e4_0_readout.md` back to Claude (and `e4_0_readout.json` if asked). The base `B` is
mechanical (§11.4): nothing is re-run, re-seeded or re-tuned after reading it. E4a (the SG-NM gate) is a separate
notebook, built on `B`'s checkpoints once `B` is recorded.